# 00 · Project Setup (Hannah)

Sets up the repository structure, shared config, and dependency files used by every other notebook.

**Run order:** `00 → 01 → 02 → 03 / 04 / 05`

**Ownership**

| File / folder | Owner |
|---|---|
| `src/config.py`, `src/pipeline.py` | Hannah |
| `src/models/stn.py`, `src/models/resnet18.py` | Hannah |
| `src/data/` (GTSRB download, preprocessing, splits, loaders) | **Alex** |
| `src/models/basic_cnn.py`, `src/models/cnn_attention.py` | **Alex** |
| corruption code, HOG+SVM | **Alex** |

```
traffic-sign-recognition/
├── notebooks/          # these notebooks
├── src/
│   ├── config.py       # Hannah
│   ├── pipeline.py     # Hannah (train / evaluate)
│   ├── data/           # ALEX
│   └── models/
│       ├── stn.py          # Hannah
│       ├── resnet18.py     # Hannah
│       ├── basic_cnn.py    # ALEX
│       └── cnn_attention.py# ALEX
├── checkpoints/  results/  figures/  data/
```

In [ ]:
# --- Locate the project root so `src/` imports work from any folder ---
import sys, os
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# --- Create the folder structure ---
for d in ["notebooks", "src", "src/models", "checkpoints", "results", "figures", "data"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

# Empty package markers. (Alex: if you also create these, an empty file is all that's needed.)
(ROOT / "src" / "__init__.py").touch()
(ROOT / "src" / "models" / "__init__.py").touch()
# NOTE: src/data/ and its __init__.py are created by ALEX's dataset code.

In [ ]:
# --- Repo housekeeping files ---
(ROOT / ".gitignore").write_text("""__pycache__/
.ipynb_checkpoints/
*.pyc
data/
checkpoints/
.venv/
.DS_Store
""")

(ROOT / "requirements.txt").write_text("""torch>=2.3
torchvision>=0.18
numpy
scikit-learn
matplotlib
tqdm
pandas
opencv-python        # likely needed by Alex's corruption code
scikit-image         # likely needed by Alex's HOG + SVM baseline
""")

(ROOT / "README.md").write_text("""# Traffic Sign Recognition with a Lightweight Attention-Augmented CNN

Florida Polytechnic University - Hannah Vutuan & Alexandra Hyatali

Run notebooks in `notebooks/` in numeric order. Shared code lives in `src/`.
All models are trained with the same `TrainConfig` (src/config.py), seed, and data splits.
""")
print("Wrote .gitignore, requirements.txt, README.md")

In [ ]:
# --- Initialise git (safe to re-run) ---
import subprocess
if not (ROOT / ".git").exists():
    subprocess.run(["git", "init"], cwd=ROOT)
    subprocess.run(["git", "add", "."], cwd=ROOT)
    subprocess.run(["git", "commit", "-m", "Initial project structure"], cwd=ROOT)
# Then add the remote and push from a terminal:
#   git remote add origin <your-github-url>
#   git push -u origin main

## Shared training configuration

Every model (HOG+SVM excluded) is trained with the same `TrainConfig`, so the ablation study is a fair comparison. Change values here, not inside individual notebooks.

In [ ]:
%%writefile src/config.py
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional

import torch

ROOT = Path(__file__).resolve().parent.parent
CHECKPOINT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"          # used by ALEX's data loaders
for _d in (CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR):
    _d.mkdir(exist_ok=True)

NUM_CLASSES = 43


@dataclass
class TrainConfig:
    seed: int = 42
    num_classes: int = NUM_CLASSES
    in_channels: int = 3
    # img_size MUST match the resize used in ALEX's preprocessing / data loaders.
    img_size: int = 48

    batch_size: int = 128
    epochs: int = 30
    lr: float = 1e-3
    weight_decay: float = 1e-4
    optimizer: str = "adamw"       # adamw | adam | sgd
    scheduler: str = "cosine"      # cosine | step | none
    label_smoothing: float = 0.0
    grad_clip: Optional[float] = 1.0
    early_stop_patience: int = 7   # epochs without val macro-F1 improvement
    num_workers: int = 2
    amp: bool = True               # mixed precision (CUDA only)

    def to_dict(self):
        return asdict(self)


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [ ]:
from src.config import TrainConfig, get_device
cfg = TrainConfig()
print(cfg)
print("Device:", get_device())

## Interfaces I'm expecting from Alex

My notebooks import the following. Nothing below is implemented here — it is **Alex's part**.

```python
# src/data/__init__.py   (ALEX)
get_dataloaders(batch_size, num_workers, img_size) -> (train_loader, val_loader, test_loader)
#   * batches are (images, labels); images float32 in [0, 1], shape (B, 3, img_size, img_size)
#   * labels are ints 0..42
#   * same train/val/test split for every model

# src/models/basic_cnn.py    (ALEX)   BasicCNN(num_classes)    -> nn.Module, (B,3,H,W) -> logits (B,num_classes)
# src/models/cnn_attention.py (ALEX)  CNNAttention(num_classes) -> same interface, CNN with channel attention
```

If Alex's interfaces differ, only the clearly marked `# ALEX'S PART` lines in notebooks 03–05 need adjusting.